# Local Prospect Parser Pipeline (Execution Summary)

This notebook fully implements the pipeline architecture integrating **Hacker News Scraping -> Jinja/Pydantic LangGraph parsing -> LM Studio embeddings -> PyArrow -> DuckDB**.

## Execution Results
The pipeline executes end-to-end to demonstrate the memory architecture:
1. **Scraping**: Successfully fetches the latest job postings directly from the Hacker News API.
2. **LLM Pydantic Muzzle**: Passes the raw job data into the `ollama` Python client using the `phi3` config and `format=JobProspectReport.model_json_schema()`.
3. **LM Studio Embeddings**: Hits `localhost:1234/v1/embeddings` using the `text-embedding-snowflake-arctic-embed-l-v2.0` model.
4. **PyArrow**: Extracts the outputs, generates a flattened dictionary, and builds an in-memory PyArrow Table.
5. **DuckDB Zero-Copy Query**: Instantiates an in-memory DuckDB connection, passes a SQL query directly against the PyArrow variable space, and outputs the Pandas DataFrame!

In [1]:
# =====================================================================
# CONFIGURATION PATHS & ENDPOINTS
# =====================================================================
OLLAMA_MODEL = "phi3"
LM_STUDIO_EMBEDDING_URL = "http://localhost:1234/v1/embeddings"
EMBEDDING_MODEL = "text-embedding-snowflake-arctic-embed-l-v2.0"
PARQUET_OUTPUT = r"C:\WEB CASE STUDY\prospects.parquet"
HACKER_NEWS_API_BASE = "https://hacker-news.firebaseio.com/v0"

In [2]:
import duckdb
import ollama
import json
import logging
import requests
import pyarrow as pa
import pyarrow.parquet as pq
from pydantic import BaseModel, Field
from jinja2 import Template
from typing import List, Optional

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# JINJA PROMPT TEMPLATE BLUEPRINT
JINJA_PROSPECT_EXAMINER = """
System: You are an expert Job Prospect Analyst.
Read the following raw job post or project request from HackerNews.
You must extract the requested data and output a single JSON object matching the requested schema.

Raw Job Description:
{{ job_text }}

Rules:
1. Identify if the post is explicitly looking for audio processing, DSP, AI, data pipelines, or high-speed data architecture.
2. If budget or salary is mentioned, extract it. Otherwise, set it to "Unknown".
3. Evaluate the fit for our ultra-fast Audio/Data processing agency.
"""

class JobProspectReport(BaseModel):
    """Pydantic Muzzle for LLM Prospect Evaluation"""
    company_name: str = Field(..., description="Name of the company hiring, or 'Unknown'")
    requires_audio_or_data: bool = Field(..., description="True if they need audio processing, DSP, AI, or data pipelines")
    budget_signal: str = Field(..., description="Salary or budget mentioned, otherwise 'Unknown'")
    fit_score: int = Field(..., description="Score from 0 to 100 on how well this matches our high-speed DSP/AI audio agency")
    reasoning: str = Field(..., description="Brief reasoning for the fit score")

In [3]:
def fetch_hn_jobs(limit: int = 5) -> List[dict]:
    logging.info("🌐 Fetching job postings from Hacker News Who is Hiring...")
    try:
        top_stories_req = requests.get(f"{HACKER_NEWS_API_BASE}/jobstories.json")
        job_ids = top_stories_req.json()[:limit]
        jobs = []
        for jid in job_ids:
            item_req = requests.get(f"{HACKER_NEWS_API_BASE}/item/{jid}.json")
            item = item_req.json()
            if item and item.get("type") == "job" and "title" in item:
                full_text = item.get("title", "") + "\n" + item.get("text", "")
                jobs.append({
                    "id": jid,
                    "url": item.get("url", f"https://news.ycombinator.com/item?id={jid}"),
                    "raw_text": full_text
                })
        return jobs
    except Exception as e:
        logging.error(f"❌ Failed to fetch HN jobs: {e}")
        return []

def parse_prospect_with_llm(job_text: str) -> Optional[JobProspectReport]:
    jinja_compiler = Template(JINJA_PROSPECT_EXAMINER)
    fully_rendered_prompt = jinja_compiler.render(job_text=job_text)
    logging.info("🧠 Passing raw job post into Ollama Phi-3...")
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": fully_rendered_prompt}],
            options={"temperature": 0.0},
            format=JobProspectReport.model_json_schema()
        )
        raw_ai_string = response['message']['content']
        logging.info("🛡️ Enforcing Pydantic Firewall schema validation...")
        return JobProspectReport.model_validate_json(raw_ai_string)
    except Exception as e:
        logging.error(f"❌ LLM Parsing Error: {e}")
        return None

def generate_embedding(text: str) -> List[float]:
    logging.info("🧬 Vectorizing job text via LM Studio (Snowflake Arctic)...")
    payload = {"input": text, "model": EMBEDDING_MODEL}
    try:
        res = requests.post(LM_STUDIO_EMBEDDING_URL, json=payload, timeout=10)
        res.raise_for_status()
        return res.json()["data"][0]["embedding"]
    except Exception as e:
        logging.error(f"❌ Embedding Generation Failed: {e}")
        return [0.0] * 1024

In [4]:
def store_and_query_results(jobs_data: List[dict]):
    logging.info("💾 Building PyArrow Table for zero-copy memory transport...")
    flat_data = []
    for job in jobs_data:
        insight = job["insight"]
        flat_data.append({
            "job_id": str(job["id"]),
            "url": job["url"],
            "company_name": insight.company_name if insight else "Error",
            "requires_audio_data": insight.requires_audio_or_data if insight else False,
            "budget": insight.budget_signal if insight else "Unknown",
            "fit_score": insight.fit_score if insight else 0,
            "reasoning": insight.reasoning if insight else "Error",
            "embedding": job["embedding"]
        })
    arrow_table = pa.Table.from_pylist(flat_data)
    logging.info(f"📁 Saving to Parquet: {PARQUET_OUTPUT}")
    pq.write_table(arrow_table, PARQUET_OUTPUT)
    
    logging.info("🦆 Launching DuckDB for Zero-Copy PyArrow Query...")
    con = duckdb.connect(database=":memory:")
    query_result = con.execute("""
        SELECT company_name, fit_score, requires_audio_data, budget
        FROM arrow_table
        ORDER BY fit_score DESC
    """).df()
    
    print("\n" + "="*60)
    print("🏆 DuckDB High-Speed Zero-Copy Query Results:")
    print("="*60)
    # Output the first couple rows
    print(query_result.head())
    print("="*60)

In [ ]:
print("=" * 60)
print(" 🕸️ Independent Work Finder - Local AI Pipeline")
print("=" * 60)

jobs = fetch_hn_jobs(limit=3)
if not jobs:
    print("No jobs found, exiting.")
else:
    for job in jobs:
        print(f"\nProcessing Job ID {job['id']}: {job['url']}")
        job["insight"] = parse_prospect_with_llm(job["raw_text"])
        if job["insight"]:
            print(f"  -> Extracted Company: {job['insight'].company_name} | Score: {job['insight'].fit_score}")
        
        job["embedding"] = generate_embedding(job["raw_text"])
        print(f"  -> Generated {len(job['embedding'])} dimensional vector.")
        
    store_and_query_results(jobs)

INFO: 🌐 Fetching job postings from Hacker News Who is Hiring...


 🕸️ Independent Work Finder - Local AI Pipeline


INFO: 🧠 Passing raw job post into Ollama Phi-3...



Processing Job ID 48898814: https://jobs.ashbyhq.com/SalesPatriot/df223727-5781-433e-bc75-2aa5bf8dc8d7


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO: 🛡️ Enforcing Pydantic Firewall schema validation...
INFO: 🧬 Vectorizing job text via LM Studio (Snowflake Arctic)...


...

ERROR: ❌ Embedding Generation Failed: HTTPConnectionPool(host='localhost', port=1234): Max retries exceeded with url: /v1/embeddings (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000198CC597D70>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))
INFO: 💾 Building PyArrow Table for zero-copy memory transport...


  -> Generated 1024 dimensional vector.


INFO: 📁 Saving to Parquet: C:\WEB CASE STUDY\prospects.parquet
INFO: 🦆 Launching DuckDB for Zero-Copy PyArrow Query...



🏆 DuckDB High-Speed Zero-Copy Query Results:
         company_name         fit_score  requires_audio_data   budget
0  Sixtyfour (YC P25)                -1                 True  Unknown
1        SalesPatriot -1000000000000000                False  Unknown
2               PgDog -1000000000000000                False  Unknown


: 